# Tool · Export an ELAN tier to a table

**What this notebook does:** It opens every ELAN file (`.eaf`) in a folder and saves the annotations of one tier
as a `.csv` table, which you can open in Excel, R or SPSS.

| You need | You get |
|---|---|
| ELAN files in a folder (by default the `Elan` folder) | One table per ELAN file, named `<file>_<tier>_annotations.csv`, with the columns `Start Time`, `End Time` (in milliseconds) and `Annotation`. |

*Notebook 3b reads ELAN files directly, so you don't need this tool for the main workflow.*

## Step 1 · Load the tools

In [ ]:
from pathlib import Path
import pandas as pd
from pympi import Elan   # reads ELAN (.eaf) files

## Step 2 · Settings

In [ ]:
ELAN_FOLDER = "../Elan"            # the folder with your .eaf files
OUTPUT_FOLDER = "../Annotations"   # where the tables go
TIER_NAME = "Stroke"               # the tier to export, for example "Manual Gesture Phasing"

## Step 3 · Export

In [ ]:
output_folder = Path(OUTPUT_FOLDER)
output_folder.mkdir(exist_ok=True)

for elan_path in sorted(Path(ELAN_FOLDER).glob("*.eaf")):
    if elan_path.stem.endswith("_backup"):
        continue  # backups made by Notebook 5
    elan = Elan.Eaf(str(elan_path))
    if TIER_NAME not in elan.get_tier_names():
        tiers = ", ".join(elan.get_tier_names())
        print(f"{elan_path.name}: no tier called '{TIER_NAME}' (its tiers are: {tiers})")
        continue

    table = pd.DataFrame(elan.get_annotation_data_for_tier(TIER_NAME),
                         columns=["Start Time", "End Time", "Annotation"]).sort_values("Start Time")
    output_path = output_folder / f"{elan_path.stem}_{TIER_NAME}_annotations.csv"
    table.to_csv(output_path, index=False)
    print(f"{elan_path.name}: {len(table)} annotations saved to {output_path}")